## Pre-tutorial steps:

### Download conda env using:
`conda env create -f environment.yml`
run this notebook using `aws-agb-env` conda env

# CTrees' Above Ground Biomass Data Access Tutorial — Summary

This tutorial shows how to access, visualize, and analyze global 100m Above Ground Biomass (AGB) data from CTrees  (hosted on the AWS Open Registry) using Python. [Additonal AGB 100m Dataset Documentation in Earthmover](https://app.earthmover.io/marketplace/69e00e1c21faca8bf36879d2)

## Data Formats

### Zarr
[Zarr](https://zarr.dev) is a format for storing chunked, compressed, N-dimensional arrays — designed from the ground up for cloud object storage. Instead of a single monolithic file (like HDF5), a Zarr store is a directory of small chunk files. This means you can read any spatial or temporal subset without downloading the full dataset; the object store serves only the specific chunks your query touches.

### Icechunk
[Icechunk](https://icechunk.io) is a transactional storage engine built on top of Zarr. It adds Git-like version control to array datasets:

- **Branches** — like Git branches, `main` is the default authoritative version of the data. Dataset authors can maintain additional branches for experiments or alternative processing runs.
- **Snapshots (commits)** — every update to the store is recorded as a snapshot, making the dataset fully reproducible at any point in time. 
- **Sessions** — a `readonly_session` pins all of your reads to a single consistent snapshot, even if the dataset is being updated concurrently.

You do not need to understand Icechunk's internals to use this data. The workflow is simply: open the repo → open a session → pass `session.store` to `xr.open_zarr()`, exactly as you would with any Zarr store. Icechunk just allows us to track the changes and you to open the most recent update while accessing it. 

### Recommended tooling

| Purpose | Library |
|---|---|
| Open & slice N-dimensional arrays | `xarray` |
| Connect to the Icechunk repository | `icechunk` |
| Reproject / clip / export rasters | `rioxarray` |
| Work with vector geometries | `geopandas` |

## Dataset Organization

Once the repository is open, its contents are structured as follows:

```
s3://ctrees-agb-100m-global/agb_100m_global   ← Icechunk repository
└── branch: main
    └── aboveground_biomass/          ← Zarr group (pass as group= to xr.open_zarr)
        ├── agb   (time, y, x)  int16 ← main data variable
        │   └── chunks: (1, 4000, 4000) — one full year per chunk
        ├── uncertainty   (time, y, x)  int16 ← main data variable
        │   └── chunks: (1, 4000, 4000) — one full year per chunk
        └── spatial_ref               ← CRS metadata (scalar variable)
```

**Key things to know:**

- **Branch `main`** — the current, authoritative version of the data. Open it with `repo.readonly_session(branch="main")`.
- **`aboveground_biomass` group** — the Zarr group you pass to `xr.open_zarr(..., group="aboveground_biomass")`. All variables live inside this group.
- **`agb` variable** — raw `int16` pixel values. Divide by the `agb_scale_factor` attribute (10) to convert to Mg ha⁻¹.
- **`uncertainty` variable** — raw `int16` pixel values. Divide by the `agb_scale_factor` attribute (10) to convert to Mg ha⁻¹.
- **Chunking `(1, 4000, 4000)`** — one complete year per chunk in the time dimension (~32 MB each). Reading a small bounding box for one year is very fast; loading all years globally at once is expensive. Keep spatial subsets small when iterating.
- **No-data value: −9999** — mask this out before computing any statistics (as shown in the cells below).

In [12]:
import geopandas as gpd
import xarray as xr
import rioxarray as rio
import icechunk

In [13]:
# Access data from the IceChunk Repo AWS open Registry
storage = icechunk.s3_storage(
    bucket="ctrees-agb-100m-global",
    prefix="agb_100m_global",
    region="us-west-2",
    anonymous=True
)

repo = icechunk.Repository.open(storage)

# View FULL AGB Dataset

In [ ]:
session = repo.readonly_session(branch="main")
ds = xr.open_zarr(session.store, zarr_format=3, group="aboveground_biomass")
ds

## Take a closer look into the "agb" variable

*   The time dimension tells you that this dataset goes from 2000-2025
*   The x & y dimension tell you this is a global dataset
*   The spatial ref tells you the projection system the data is in
*   The attributes tells you the units
*   There is a scale factor of 10


In [ ]:
ds.agb

# View clipped AGB Dataset

* Let's find a [bounding box](https://bboxfinder.com/) to look closer at the data! Replace the `bbox` below with your choice of WGS84 coordinates

In [ ]:
bbox = [-77.124023,-1.318243,-75.454102,-0.357053]

In [ ]:
minx, miny, maxx, maxy = bbox

da_clip = ds.agb.sel(
    x=slice(minx, maxx),
    y=slice(maxy, miny),
) / ds.agb.attrs.get("agb_scale_factor")

## Let's plot a few years!

In [ ]:
year_one = 2000
year_two = 2025

In [ ]:
da_clip.sel(time=f"{year_one}-01-01").plot()


In [ ]:
print(year_two)
da_clip.sel(time=f"{year_two}-01-01").plot()

## What if we wanted to save the output as a geotiff?

In [ ]:
da_year = da_clip.sel(time=f"{year_one}-01-01")

da_year.rio.to_raster(f"my_bbox_{year_one}-01-01.tif",
    tiled=True,       # better layout for partial reads
    windowed=True,    # avoids loading everything into memory at once
    compress="LZW",
)

# What if we wanted to plot a geometry?
1. upload geometry to that "files" folder you just downloaded the geotiff from
2. rename "file_name" to your geometry file name, and run the section below!

In [ ]:
file_name = "GADM41_USA.5.19_1.geojson"

In [ ]:
# --- Load geometries ---
gdf = gpd.read_file(file_name)
gdf.geometry.plot()

In [ ]:
geom = gdf.geometry
minx, miny, maxx, maxy = geom.total_bounds

# Keep all years, but crop spatially before clipping to the polygon.
da_clip = ds.agb.sel(
    x=slice(minx, maxx),
    y=slice(maxy, miny),
) / ds.agb.attrs.get("agb_scale_factor")

da_masked = da_clip.rio.clip(geom, crs="EPSG:4326", drop=True)

In [ ]:
da_masked.sel(time=f"{year_one}-01-01").plot()

In [ ]:
da_masked.sel(time=f"{year_two}-01-01").plot()

## Find AGB values for all years for clipped geometry!

In [ ]:
# Define no_data_val and apply masking
no_data_val = -9999

# Apply masking for no_data_val and negatives, then scale
cleaned = da_masked.where(da_masked != no_data_val)  # Mask no_data_val
cleaned = cleaned.where(cleaned > 0, 0)

# Compute total and average AGB for all years at once
stats_df = xr.Dataset(
    data_vars={
        "avg_agb_density": cleaned.mean(dim=("y", "x"), skipna=True).fillna(0),
    }
)

# Convert to pandas DataFrame if needed
stats_df = stats_df.to_dataframe().reset_index()

In [ ]:
## plot & download time series of average AGB density
stats_df.plot(x="time", y="avg_agb_density", title="Average Above Ground Biomass Density")

In [ ]:
stats_df.to_csv("average_agb_density_timeseries.csv", index=False)

---
## Q1 — What is one question you have answered using these data?

**Question:** Has mean above-ground biomass declined in a deforestation-affected part of the Brazilian Amazon between 2000 and 2025, and if so, by how much?

**Why this region?**  
Pará state, Brazil is one of the most-documented deforestation frontiers on Earth. Annual INPE PRODES alerts and Global Forest Watch both record sustained forest loss here across the 25-year period covered by the CTrees dataset, making it an ideal test case.

**Approach:**
1. Clip the global AGB cube to a bounding box in Pará state.
2. Mask no-data pixels and compute the spatial mean AGB for every year in the time series.
3. Plot the resulting 26-point time series and print the overall percentage change.

A sustained downward trend, particularly after 2004 (peak deforestation) and again around 2019–2022 (policy rollback period), confirms that the AGB signal tracks known land-use history in the region.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Bounding box for a deforestation-affected area in Pará state, Brazil
amazon_bbox = [-54.9, -7.5, -52.1, -5.5]
minx, miny, maxx, maxy = amazon_bbox

da_amazon = ds.agb.sel(
    x=slice(minx, maxx),
    y=slice(maxy, miny),
) / ds.agb.attrs.get("agb_scale_factor")

# Mask no-data and negative values, then compute mean AGB per year
no_data_val = -9999
da_amazon_clean = da_amazon.where(da_amazon != no_data_val).where(da_amazon > 0)

mean_agb = da_amazon_clean.mean(dim=("y", "x"), skipna=True).compute()

# Build a tidy DataFrame
ts = pd.DataFrame({
    "year": mean_agb.time.dt.year.values,
    "mean_agb_Mg_ha": mean_agb.values,
})

# Plot
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts["year"], ts["mean_agb_Mg_ha"], marker="o", linewidth=2, color="forestgreen")
ax.fill_between(ts["year"], ts["mean_agb_Mg_ha"], alpha=0.15, color="forestgreen")
ax.set_xlabel("Year")
ax.set_ylabel("Mean AGB (Mg ha⁻¹)")
ax.set_title("Mean Above-Ground Biomass — Pará, Brazil (2000–2025)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summarise the overall change
agb_2000 = ts.loc[ts["year"] == 2000, "mean_agb_Mg_ha"].values[0]
agb_2025 = ts.loc[ts["year"] == 2025, "mean_agb_Mg_ha"].values[0]
pct_change = (agb_2025 - agb_2000) / agb_2000 * 100
print(f"Mean AGB 2000 : {agb_2000:.1f} Mg ha⁻¹")
print(f"Mean AGB 2025 : {agb_2025:.1f} Mg ha⁻¹")
print(f"Overall change: {pct_change:+.1f}%")

---
## Q2 — What is one unanswered question that could be answered using these data?

**Question:** Where and when did the largest single-year drops in above-ground biomass occur, and can those disturbance events be linked to specific drivers such as fire or agricultural clearing?

**Why this matters:**  
The CTrees dataset has 25 years of annual, 100 m AGB estimates globally. A sudden large drop in AGB at a pixel (e.g., > 30 Mg ha⁻¹ in one year) is a strong signal of a disturbance event — clear-cutting, fire, storm damage, or landslide. Mapping *where* and *when* these drops occurred, and then joining the results to external datasets (INPE PRODES, NASA FIRMS fire radiative power, Global Forest Watch tree cover loss), could:

1. **Validate** the AGB signal against independently detected disturbances.
2. **Characterise** the carbon impact by disturbance type.
3. **Identify** regions where AGB is recovering after a disturbance, separating regrowth from intact forest.

**Recommendations for someone tackling this:**

- **Use Dask for large-area analysis.** Open the dataset with explicit chunks so computations are distributed:
  ```python
  ds = xr.open_zarr(session.store, zarr_format=3, group="aboveground_biomass",
                    chunks={"time": 1, "y": 2000, "x": 2000})
  ```
- **Compute year-over-year differences** with `xr.DataArray.diff(dim="time")`, then threshold to isolate large loss events.
- **Join with fire/deforestation data.** NASA FIRMS (MODIS/VIIRS) fire detections are freely available; INPE PRODES provides annual deforestation polygons for the Brazilian Amazon. Both can be loaded with `geopandas` and intersected with your loss-event raster.
- **Consider Icechunk snapshots** if you want to compare different processing versions of the CTrees data — each snapshot is addressable and reproducible.

The starter code below demonstrates the year-over-year differencing step for the Pará subset created in Q1.

In [ ]:
# Starter: detect pixels with a large single-year AGB loss
# da_amazon_clean was computed in the Q1 cell above; re-run that cell first if needed.

# Year-over-year AGB difference (positive = gain, negative = loss)
agb_diff = da_amazon_clean.diff(dim="time")

# Identify pixels that lost > 30 Mg ha⁻¹ in ANY single year
loss_threshold = -30  # Mg ha⁻¹
large_loss_mask = (agb_diff < loss_threshold).any(dim="time").compute()

# Spatial map of disturbance hotspots
fig, ax = plt.subplots(figsize=(10, 6))
large_loss_mask.plot(ax=ax, cmap="Reds", add_colorbar=False)
ax.set_title(
    "Pixels with any single-year AGB loss > 30 Mg ha⁻¹ (2000–2025)\n"
    "Pará, Brazil — potential disturbance hotspots"
)
plt.tight_layout()
plt.show()

fraction = large_loss_mask.mean().item()
print(f"Fraction of pixels with a large-loss event: {fraction:.1%}")
print()
print("Next steps: overlay this mask with INPE PRODES deforestation polygons")
print("or NASA FIRMS fire detections to characterise the dominant disturbance type.")